In [23]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold # -> This is for Stratified K Fold split for Train and Cross_Val
from sklearn.feature_extraction.text import TfidfVectorizer  # -> TF-IDF which converts the text to numbers and also assign weights
from sklearn.linear_model import LogisticRegression # -> This will be the model
from sklearn.svm import LinearSVC # -> This is another model
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report # Metrics for testing the best model
import joblib # -> To save the model to the disk

In [2]:
dev_data = pd.read_csv('../data/processed/dev_data.csv')
X_dev = dev_data['text']
y_dev = dev_data['sentiment']

In [3]:
#Let's create a 5-fold stratified cross-validation set
skf = StratifiedKFold(n_splits=5, shuffle=True,random_state=42)


In [4]:
# Empty list to store the log_reg results for each fold
logreg_fold_results = []
linear_fold_results = []
# Manual 5 fold training loop
for fold, (train_idx, val_idx) in enumerate(skf.split(X_dev,y_dev),start=1):
    
    # 1. Create train and val data from each fold
    X_train = X_dev.iloc[train_idx]
    y_train = y_dev.iloc[train_idx]

    X_val = X_dev.iloc[val_idx]
    y_val = y_dev.iloc[val_idx]

    # 2. Create the TF_IDF object
    tfidf = TfidfVectorizer()
    # fit - Learns vocubulary and learns IDF values from the input
    # transform -> converts text to TF-IDF number
    
    # 3. TF-IDF fit only on the train_fold
    X_train_tfidf = tfidf.fit_transform(X_train)
    
    # 4. Transform validation fold using the training fold's TF-IDF
    X_val_tfidf = tfidf.transform(X_val)
    
    # 5. Create the model
    model = LinearSVC(
        class_weight='balanced',
        random_state=42
    )

    # 6. Train model on the current fold's training data
    model.fit(X_train_tfidf, y_train)

    # 7. Predict on the validation fold
    y_val_predict = model.predict(X_val_tfidf)

    # 8. Calculate Metrics
    accuracy = accuracy_score(y_val, y_val_predict)
    precision_macro = precision_score(y_val, y_val_predict, average='macro')
    recall_macro = recall_score(y_val, y_val_predict,average='macro')
    f1_macro = f1_score(y_val, y_val_predict, average='macro')

    # 9. Store the results
    linear_fold_results.append(
        {
            'fold': fold,
            'accuracy': accuracy,
            'precision_macro': precision_macro,
            'recall_macro': recall_macro,
            'f1_macro': f1_macro
        }
    )
    # 10. Print fold result
    print(f"Fold {fold}")
    print("Train size:", len(X_train))
    print("Validation size:", len(X_val))
    print("Accuracy:", accuracy)
    print("Macro F1:", f1_macro)
    print("-" * 50)

Fold 1
Train size: 3272
Validation size: 819
Accuracy: 0.7777777777777778
Macro F1: 0.7355099951397664
--------------------------------------------------
Fold 2
Train size: 3273
Validation size: 818
Accuracy: 0.7677261613691931
Macro F1: 0.7238360188372411
--------------------------------------------------
Fold 3
Train size: 3273
Validation size: 818
Accuracy: 0.7591687041564792
Macro F1: 0.7052576619426011
--------------------------------------------------
Fold 4
Train size: 3273
Validation size: 818
Accuracy: 0.7811735941320294
Macro F1: 0.7447962949195644
--------------------------------------------------
Fold 5
Train size: 3273
Validation size: 818
Accuracy: 0.7787286063569682
Macro F1: 0.7373765856015103
--------------------------------------------------


In [5]:
# Let's convert the results into a dataframe
logreg_results_df = pd.DataFrame(logreg_fold_results)
logreg_results_df

""


In [6]:
# Get the average CV results
logreg_summary = {
    "model": "TF-IDF + Logistic Regression",
    "accuracy_mean": logreg_results_df["accuracy"].mean(),
    "accuracy_std": logreg_results_df["accuracy"].std(),
    "macro_precision_mean": logreg_results_df["precision_macro"].mean(),
    "macro_recall_mean": logreg_results_df["recall_macro"].mean(),
    "macro_f1_mean": logreg_results_df["f1_macro"].mean(),
    "macro_f1_std": logreg_results_df["f1_macro"].std()
}

logreg_summary

KeyError: 'accuracy'

In [7]:
linear_model_results_df = pd.DataFrame(linear_fold_results)
linear_model_results_df

,fold,accuracy,precision_macro,recall_macro,f1_macro
0,1,0.777778,0.743413,0.728941,0.735510
1,2,0.767726,0.731444,0.718199,0.723836
2,3,0.759169,0.721378,0.692487,0.705258
3,4,0.781174,0.783504,0.718063,0.744796
4,5,0.778729,0.743133,0.732440,0.737377


In [8]:
# Get the average CV results
svm_summary = {
    "model": "TF-IDF + Linear SVM",
    "accuracy_mean": linear_model_results_df["accuracy"].mean(),
    "accuracy_std": linear_model_results_df["accuracy"].std(),
    "macro_precision_mean": linear_model_results_df["precision_macro"].mean(),
    "macro_recall_mean": linear_model_results_df["recall_macro"].mean(),
    "macro_f1_mean": linear_model_results_df["f1_macro"].mean(),
    "macro_f1_std": linear_model_results_df["f1_macro"].std()
}

svm_summary

{'model': 'TF-IDF + Linear SVM',
 'accuracy_mean': np.float64(0.7729149687584895),
 'accuracy_std': np.float64(0.009240315191819634),
 'macro_precision_mean': np.float64(0.7445743903860282),
 'macro_recall_mean': np.float64(0.7180262545226856),
 'macro_f1_mean': np.float64(0.7293553112881367),
 'macro_f1_std': np.float64(0.015425662413400801)}

In [9]:
compare_df = pd.DataFrame([
    logreg_summary,
    svm_summary
])
compare_df.round(4)

NameError: name 'logreg_summary' is not defined

*The **TF-IDF + Linear SVM model** achieved the strongest cross-validation performance, with a **mean accuracy of 77.29%** and a **mean macro F1 score of 72.94%.** It outperformed **TF-IDF + Logistic Regression,** which achieved **75.68% accuracy and 71.48% macro F1**. Since macro F1 is the main metric for this imbalanced sentiment dataset, Linear SVM is selected as the best classical baseline model.*

In [10]:
best_model = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('model',LinearSVC(
        class_weight='balanced',
        random_state=42
    ))
])

In [11]:
best_model.fit(X_dev,y_dev)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['negative','neutral','positive']"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True


In [12]:
# Let's load the test data once
test_data = pd.read_csv('../data/processed/test_data.csv')
X_test = test_data['text']
y_test = test_data['sentiment']

In [13]:
# Let's check how did the model predict
y_pred = best_model.predict(X_test)

In [14]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test Macro F1:", f1_score(y_test, y_pred, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Test Accuracy: 0.7441217150760719
Test Macro F1: 0.7076800178390671

Classification Report:
              precision    recall  f1-score   support

    negative       0.66      0.71      0.69        91
     neutral       0.79      0.82      0.81       428
    positive       0.66      0.59      0.63       204

    accuracy                           0.74       723
   macro avg       0.71      0.71      0.71       723
weighted avg       0.74      0.74      0.74       723


Confusion Matrix:
[[ 65  19   7]
 [ 22 352  54]
 [ 11  72 121]]


*The selected TF-IDF + Linear SVM model achieved 74.41% accuracy and 70.77% macro F1 on the final untouched test set. This significantly outperforms the majority-class baseline of 59.16% accuracy. The model performs best on the neutral class, with an F1-score of 0.81. The negative class also performs reasonably well despite being the minority class, with recall of 0.71. The weakest class is positive, with recall of 0.59, indicating that many positive headlines are being predicted as neutral. The confusion matrix shows that the largest error pattern is confusion between positive and neutral sentiment.*

**#ERROR ANALYSIS**

In [15]:
test_results = test_data.copy()

test_results["predicted_sentiment"] = y_pred

test_results["is_correct"] = (
    test_results["sentiment"] == test_results["predicted_sentiment"]
)

test_results["error_type"] = (
    test_results["sentiment"] + " -> " + test_results["predicted_sentiment"]
)

wrong_predictions = test_results[test_results["is_correct"] == False].copy()

error_analysis_sample = (
    wrong_predictions
    .groupby("error_type", group_keys=False)
    .sample(n=4, random_state=42)
)

error_analysis_sample[
    ["original_index", "text", "sentiment", "predicted_sentiment", "error_type"]
]


,original_index,text,sentiment,predicted_sentiment,error_type
9,4341,Border Guard Service has banned the mooring of...,negative,neutral,negative -> neutral
195,4543,Small investors have voiced fears that the sha...,negative,neutral,negative -> neutral
489,3472,A total of 140 jobs will be reduced at the Raa...,negative,neutral,negative -> neutral
85,3295,The personnel reduction will be carried out in...,negative,neutral,negative -> neutral
628,4413,Cash flow from operations in January-December ...,negative,positive,negative -> positive
339,4811,Operating profit fell to EUR 35.4 mn from EUR ...,negative,positive,negative -> positive
701,1706,TeliaSonera 's underlying results however incl...,negative,positive,negative -> positive
83,4544,The result will also be burdened by increased ...,negative,positive,negative -> positive
312,4189,"( ADPnews ) - Dec 30 , 2009 - Finnish investme...",neutral,negative,neutral -> negative
613,2401,"At 10.58 am , Outokumpu declined 2.74 pct to 2...",neutral,negative,neutral -> negative


In [16]:
# Since our text column is truncated 
pd.set_option("display.max_colwidth", None)

error_analysis_sample[
    ["original_index", "text", "sentiment", "predicted_sentiment", "error_type"]
]

,original_index,text,sentiment,predicted_sentiment,error_type
9,4341,Border Guard Service has banned the mooring of the company car-shipping ferry on its test travel at the railroad-car terminal of the Russian port as the border checkpoint is not yet ready .,negative,neutral,negative -> neutral
195,4543,Small investors have voiced fears that the shares will end up with risk investors .,negative,neutral,negative -> neutral
489,3472,A total of 140 jobs will be reduced at the Raahe Steel Works .,negative,neutral,negative -> neutral
85,3295,"The personnel reduction will be carried out in Anjalankoski , Hollola , Jyvaskyla , Jarvenpaa , Karhula , Turku and Valkeakoski units .",negative,neutral,negative -> neutral
628,4413,Cash flow from operations in January-December 2008 was a negative EUR 18.1 mn compared to EUR 39.0 mn in the corresponding period in 2007 .,negative,positive,negative -> positive
339,4811,"Operating profit fell to EUR 35.4 mn from EUR 68.8 mn in 2007 , including vessel sales gain of EUR 12.3 mn .",negative,positive,negative -> positive
701,1706,"TeliaSonera 's underlying results however included 457 mln skr in positive one-offs , hence the adjusted underlying EBITDA actually amounts to 7.309 bln skr , clearly below expectations , analysts said .",negative,positive,negative -> positive
83,4544,"The result will also be burdened by increased fixed costs associated with operations in China , and restructuring costs in Japan .",negative,positive,negative -> positive
312,4189,"( ADPnews ) - Dec 30 , 2009 - Finnish investment group Neomarkka Oyj ( HEL : NEMBV ) said today that it will furlough employee in its unit Reka Cables Ltd for less than 90 days , starting in January 2010 .",neutral,negative,neutral -> negative
613,2401,"At 10.58 am , Outokumpu declined 2.74 pct to 24.87 eur , while the OMX Helsinki 25 was 0.55 pct higher at 2,825.14 and the OMX Helsinki added 0.64 pct to 9,386.89 .",neutral,negative,neutral -> negative


In [17]:
#Let's get into the analysis where the model predicted wrong 
error_analysis_sample  = error_analysis_sample.copy()
error_analysis_sample['error_reason_category'] = ''
error_analysis_sample['reason_it_failed'] = ''

In [18]:
error_explanations = {
    4341: {
        "error_reason_category": "negative_neutral_boundary",
        "reason_it_failed": "The sentence describes an operational setback, but it is written in a factual news style. The model may not have learned that words such as 'banned' and 'not yet ready' indicate negative business impact, so it classified the headline as neutral."
    },

    4543: {
        "error_reason_category": "negative_neutral_boundary",
        "reason_it_failed": "The headline contains investor fears about share-related risk, which is negative sentiment. However, the wording is indirect and not based on common financial decline words such as loss, fall, or decrease, so the model likely treated it as neutral."
    },

    3472: {
        "error_reason_category": "negative_neutral_boundary",
        "reason_it_failed": "The sentence reports job reductions, which is a negative business event. The model likely treated it as neutral because the wording is factual and administrative rather than strongly negative."
    },

    3295: {
        "error_reason_category": "negative_neutral_boundary",
        "reason_it_failed": "The sentence describes personnel reductions across multiple units, which is negative. However, it is written in a formal factual tone, so the model likely treated it as neutral news rather than negative sentiment."
    },

    4413: {
        "error_reason_category": "missed_directionality",
        "reason_it_failed": "The model likely focused on positive-sounding finance terms such as cash flow and operations, but missed that the cash flow was negative and much worse than the previous year."
    },

    4811: {
        "error_reason_category": "missed_directionality",
        "reason_it_failed": "The headline contains the word profit, which may have pushed the model toward positive. However, the actual direction is negative because operating profit fell sharply from the previous year."
    },

    1706: {
        "error_reason_category": "mixed_financial_signal",
        "reason_it_failed": "The sentence contains positive-looking terms such as positive one-offs and EBITDA, but the actual sentiment is negative because adjusted EBITDA was clearly below expectations. The model likely overreacted to positive finance words and missed the final negative context."
    },

    4544: {
        "error_reason_category": "missed_negative_context",
        "reason_it_failed": "The model likely did not capture that burdened, increased fixed costs, and restructuring costs indicate negative pressure on results. The sentence is financially negative even though it is written in a formal factual tone."
    },

    4189: {
        "error_reason_category": "keyword_overreaction",
        "reason_it_failed": "The model likely reacted to negative words such as furlough and employee, but the dataset label treats this as neutral factual reporting. This shows that the model can over-classify factual corporate actions as negative."
    },

    2401: {
        "error_reason_category": "keyword_overreaction",
        "reason_it_failed": "The word declined strongly suggests negative sentiment, so the model predicted negative. However, the sentence is mainly a factual market-price update with broader index context, so the true label is neutral."
    },

    480: {
        "error_reason_category": "keyword_overreaction",
        "reason_it_failed": "The model likely focused on the word decreased and interpreted it as negative. In context, the sentence describes operational efficiency from better contact information, so the label is neutral rather than negative."
    },

    4587: {
        "error_reason_category": "misleading_token",
        "reason_it_failed": "The phrase low-down may have looked negative at the token level, but in this context it simply means an explanation or overview. The model lacks contextual understanding and treated the phrase as negative."
    },

    3565: {
        "error_reason_category": "neutral_positive_boundary",
        "reason_it_failed": "The sentence contains positive-sounding business terms such as global supply, solutions, and services. However, it is only describing the company's business activities, so the correct label is neutral."
    },

    1362: {
        "error_reason_category": "neutral_positive_boundary",
        "reason_it_failed": "The model may have interpreted shareholders, vote, and agreement as positive corporate-action signals. But the sentence only states that shareholders can vote on an agreement, which is factual and neutral."
    },

    1536: {
        "error_reason_category": "neutral_positive_boundary",
        "reason_it_failed": "The sentence mentions increase and growing smartphone popularity, which can look positive to a keyword-based model. However, it is a factual industry trend statement, not a direct positive company sentiment."
    },

    399: {
        "error_reason_category": "neutral_positive_boundary",
        "reason_it_failed": "The model likely treated increase its stake as a positive business signal. But the wording is conditional and factual, so the true label is neutral."
    },

    4066: {
        "error_reason_category": "missed_comparison_direction",
        "reason_it_failed": "The sentence is positive because cash flow improved from a negative EUR 68.6 mn to a positive EUR 7.4 mn. The model likely focused on the word negative and the large negative number, missing the improvement direction."
    },

    2121: {
        "error_reason_category": "missed_comparison_direction",
        "reason_it_failed": "The sentence is positive because the company moved from a loss to a pretax profit. The model likely focused on the word loss and failed to understand that the comparison shows improvement."
    },

    2167: {
        "error_reason_category": "missed_comparison_direction",
        "reason_it_failed": "The sentence is positive because pretax profit is reported compared with a large loss in the previous period. The model likely overreacted to the word loss and missed the positive turnaround."
    },

    2101: {
        "error_reason_category": "missed_positive_financial_direction",
        "reason_it_failed": "The sentence reports higher earnings before tax and higher net profit compared with the previous year. The model likely failed to interpret the numeric comparison correctly and did not capture the positive direction."
    },

    813: {
        "error_reason_category": "positive_neutral_boundary",
        "reason_it_failed": "The sentence describes a lease agreement and hotel construction, which can be a positive business development. However, it is written as a factual agreement, so the model treated it as neutral."
    },

    2253: {
        "error_reason_category": "positive_neutral_boundary",
        "reason_it_failed": "The positive signal comes from growing and rapidly internationalizing. The model may have treated the sentence as a simple company description instead of recognizing it as positive business momentum."
    },

    2015: {
        "error_reason_category": "positive_neutral_boundary",
        "reason_it_failed": "The sentence describes signing a Letter of Intent with a strategic investor, which is a positive business signal. The model likely treated it as a factual corporate announcement because the positive impact is implicit."
    },

    389: {
        "error_reason_category": "positive_neutral_boundary",
        "reason_it_failed": "The sentence describes a signed lease contract, which the dataset labels as positive. The model likely classified it as neutral because contract announcements often read like factual business news unless value or growth impact is explicit."
    }
}

In [19]:
for original_index, explanation in error_explanations.items():
    error_analysis_sample.loc[
        error_analysis_sample["original_index"] == original_index,
        "error_reason_category"
    ] = explanation["error_reason_category"]

    error_analysis_sample.loc[
        error_analysis_sample["original_index"] == original_index,
        "reason_it_failed"
    ] = explanation["reason_it_failed"]

In [21]:
error_analysis_sample[
    [
        "original_index",
        "text",
        "sentiment",
        "predicted_sentiment",
        "error_type",
        "error_reason_category",
        "reason_it_failed"
    ]
]
error_analysis_sample.shape

(24, 8)

In [22]:
error_analysis_sample.to_csv('../reports/error_analysis_sample.csv', index=False)

In [24]:
# Let's save the model to the local
joblib.dump(best_model, '../models/tfidf_linear_svm.joblib')

['../models/tfidf_linear_svm.joblib']